## Compute EFPs, Store in CSVs

Using `measure='ee'` (appropriate for e+e- events), which uses energy fractions z_i = E_i / sum(E_j) and pairwise angles theta_ij computed from the 3-momenta directions.

Input to `batch_compute`: list of (N_particles, 4) arrays with columns (E, px, py, pz).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import energyflow as ef
from energyflow.utils import gen_massless_phase_space
from energyflow import EFPSet
import os
import time
from RegressFunctions import *
from RestrictedEFPs import efps_chunked
from PhysicsFunctions import *

os.makedirs('figures', exist_ok=True)

## Load Events

In [2]:
#########################################################
### Load events from my Pythia simulation 
#########################################################
def load_events(filepath):
    events, current = [], []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line:
                current.append([float(x) for x in line.split()])
            elif current:
                events.append(np.array(current))
                current = []
    if current:
        events.append(np.array(current))
    return events

Zqq_events = load_events('data/Zqq_events.txt')
Hgg_events = load_events('data/Hgg_events.txt')


mults_Z = [len(e) for e in Zqq_events]
mults_H = [len(e) for e in Hgg_events]
print(f'Z->qqbar: {len(Zqq_events)} events, avg multiplicity {np.mean(mults_Z):.1f} +/- {np.std(mults_Z):.1f}')
print(f'H->gg:    {len(Hgg_events)} events, avg multiplicity {np.mean(mults_H):.1f} +/- {np.std(mults_H):.1f}')


#########################################################
### Load events from zqq_splittings folder 
#########################################################
def load_splittings(k, folder='data/zqq_splittings'):
    """Z->qqbar partons after the k-th shower branching.
    Returns array (N_k, 2+k, 4) with columns (E, px, py, pz)."""
    return np.load(f'{folder}/splittings_{k}.npy').astype(np.float64)

Zqq_events_1 = load_splittings(1)   # 3 partons
Zqq_events_2 = load_splittings(2)   # 4 partons
Zqq_events_3 = load_splittings(3)   # 5 partons
Zqq_events_4 = load_splittings(4)   # 6 partons

for k, ev in enumerate([Zqq_events_1, Zqq_events_2, Zqq_events_3, Zqq_events_4], start=1):
    print(f'Z->qqbar, {k} splitting(s): {ev.shape[0]} events, {ev.shape[1]} partons each')



##############################
### Uniform Phase Space #####
##############################

NEvents=100_000
uniform_N3 = gen_massless_phase_space(NEvents,3,energy=1.0)
uniform_N4 = gen_massless_phase_space(NEvents,4,energy=1.0)
uniform_N5 = gen_massless_phase_space(NEvents,5,energy=1.0)
uniform_N7 = gen_massless_phase_space(NEvents,7,energy=1.0)
uniform_N10 = gen_massless_phase_space(NEvents,10,energy=1.0)


Z->qqbar: 100000 events, avg multiplicity 12.4 +/- 4.2
H->gg:    100000 events, avg multiplicity 19.9 +/- 5.2
Z->qqbar, 1 splitting(s): 99387 events, 3 partons each
Z->qqbar, 2 splitting(s): 98800 events, 4 partons each
Z->qqbar, 3 splitting(s): 96813 events, 5 partons each
Z->qqbar, 4 splitting(s): 93358 events, 6 partons each


In [3]:
events = {'u3': uniform_N3, 'u4': uniform_N4, 'u5': uniform_N5,
          'u7': uniform_N7, 'u10': uniform_N10,
          'Z': Zqq_events, 'H': Hgg_events,
          'Z1': Zqq_events_1, 'Z2': Zqq_events_2, 'Z3': Zqq_events_3, 'Z4': Zqq_events_4}  

## Compute EFPs, Save

In [4]:
### Compute full set of EFPs 

# efpset = ef.EFPSet(('d<=', 8),  measure='eeefm', beta=2, normed=True, coords='epxpypz')
# print(f'EFP basis: {efpset.count()} EFPs')

# EFPs_d8_N3 = efpset.batch_compute(uniform_N3, n_jobs=n_jobs);   print('uniform N=3 done')
# EFPs_d8_N5 = efpset.batch_compute(uniform_N5, n_jobs=n_jobs);   print('uniform N=5 done')
# EFPs_d8_N7 = efpset.batch_compute(uniform_N7, n_jobs=n_jobs);   print('uniform N=7 done')
# EFPs_d8_N10 = efpset.batch_compute(uniform_N10, n_jobs=n_jobs);   print('uniform N=10 done')

# EFPs_d8_Zqq = efpset.batch_compute(Zqq_events, n_jobs=n_jobs); print('Z->qqbar done')
# EFPs_d8_Hgg = efpset.batch_compute(Hgg_events, n_jobs=n_jobs); print('H->gg done')


In [5]:
### Compute restricted set of EFPs  (the 3 graph types: dipole, wedge, triangle)
EFPs_restricted = {}
for key, ev in events.items():
    if key in ('Z', 'H'): continue                                   # full showers are ragged lists; skip for now
    t0 = time.time()
    EFPs_restricted[key] = efps_chunked(ev, dmax=13, device='cuda', chunk=5000).cpu().numpy()
    print(f'{key:>4}: events {ev.shape} -> EFPs {EFPs_restricted[key].shape}   ({time.time() - t0:.1f} s)')

  u3: events (100000, 3, 4) -> EFPs (100000, 109)   (0.7 s)
  u4: events (100000, 4, 4) -> EFPs (100000, 109)   (0.3 s)
  u5: events (100000, 5, 4) -> EFPs (100000, 109)   (0.3 s)
  u7: events (100000, 7, 4) -> EFPs (100000, 109)   (0.3 s)
 u10: events (100000, 10, 4) -> EFPs (100000, 109)   (0.4 s)
  Z1: events (99387, 3, 4) -> EFPs (99387, 109)   (0.2 s)
  Z2: events (98800, 4, 4) -> EFPs (98800, 109)   (0.3 s)
  Z3: events (96813, 5, 4) -> EFPs (96813, 109)   (0.3 s)
  Z4: events (93358, 6, 4) -> EFPs (93358, 109)   (0.2 s)


In [7]:
#### Save restricted EFPs as .npz (keys = u3, u4, u5, u7, u10, Z1..Z4) ####
DMAX = 15
np.savez_compressed(f'features/EFPs_restricted_d{DMAX}.npz', dmax=DMAX, **EFPs_restricted)
print({k: v.shape for k, v in EFPs_restricted.items()})

{'u3': (100000, 109), 'u4': (100000, 109), 'u5': (100000, 109), 'u7': (100000, 109), 'u10': (100000, 109), 'Z1': (99387, 109), 'Z2': (98800, 109), 'Z3': (96813, 109), 'Z4': (93358, 109)}


## Compute S_IJ's, save

In [8]:
### s_ij features: sorted pairwise invariants s_ij = (p_i + p_j)^2, i < j
def pair_inv(ev):   # (n, N, 4) -> (n, N(N-1)/2), sorted descending
    E, p = ev[..., 0], ev[..., 1:]
    s = 2*(E[:, :, None]*E[:, None, :] - np.einsum('nia,nja->nij', p, p))
    iu = np.triu_indices(ev.shape[1], 1)
    return -np.sort(-s[:, iu[0], iu[1]], axis=1)

### Compute s_ij
SIJ = {}
for key, ev in events.items():
    if key in ('Z', 'H'): continue      # ragged lists; skip for now
    t0 = time.time()
    SIJ[key] = pair_inv(ev)
    print(f'{key:>4}: events {ev.shape} -> s_ij {SIJ[key].shape}   ({time.time() - t0:.1f} s)')

  u3: events (100000, 3, 4) -> s_ij (100000, 3)   (0.1 s)
  u4: events (100000, 4, 4) -> s_ij (100000, 6)   (0.1 s)
  u5: events (100000, 5, 4) -> s_ij (100000, 10)   (0.1 s)
  u7: events (100000, 7, 4) -> s_ij (100000, 21)   (0.2 s)
 u10: events (100000, 10, 4) -> s_ij (100000, 45)   (0.3 s)
  Z1: events (99387, 3, 4) -> s_ij (99387, 3)   (0.1 s)
  Z2: events (98800, 4, 4) -> s_ij (98800, 6)   (0.1 s)
  Z3: events (96813, 5, 4) -> s_ij (96813, 10)   (0.1 s)
  Z4: events (93358, 6, 4) -> s_ij (93358, 15)   (0.1 s)


In [9]:
#### Save s_ij as .npz (keys = u3, u4, u5, u7, u10, Z1..Z4) ####
np.savez_compressed('features/SIJ.npz', **SIJ)
print({k: v.shape for k, v in SIJ.items()})

{'u3': (100000, 3), 'u4': (100000, 6), 'u5': (100000, 10), 'u7': (100000, 21), 'u10': (100000, 45), 'Z1': (99387, 3), 'Z2': (98800, 6), 'Z3': (96813, 10), 'Z4': (93358, 15)}
